# 03 Modeling Baselines

## 1. Project setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

## 2. Imports

In [ ]:
import pandas as pd

## 3. Paths and configuration

In [ ]:
from src.config import FEATURES_CSV, FIGURES_DIR, MODEL_SCORES_CSV, RANDOM_STATE, ensure_output_dirs
from src.train_models import feature_columns, make_group_labels, split_train_test, train_and_evaluate_models

ensure_output_dirs()

## 4. Load metadata

In [ ]:
features = pd.read_csv(FEATURES_CSV)
features.head()

## 5. Sanity checks

In [ ]:
y = features["label"]
groups = make_group_labels(features)
columns = feature_columns(features)

print(f"Rows: {len(features)}")
print(f"Image-derived feature columns: {len(columns)}")
print("Label counts:")
print(y.value_counts())
print(f"Sample-area groups: {groups.nunique()}")

## 6. Main analysis

Models used:
- `DummyClassifier`: majority-class baseline.
- `LogisticRegression`: linear baseline for separability of engineered features.
- `DecisionTreeClassifier`: interpretable nonlinear rules.
- `RandomForestClassifier`: more stable tree ensemble for nonlinear interactions.
- `KNeighborsClassifier`: local similarity baseline after scaling.
- `SVC`: margin-based nonlinear baseline after scaling.

In [ ]:
train_idx, test_idx = split_train_test(features, random_state=RANDOM_STATE)
train_groups = set(groups.loc[train_idx])
test_groups = set(groups.loc[test_idx])
assert train_groups.isdisjoint(test_groups)

scores = train_and_evaluate_models(
    features,
    FIGURES_DIR,
    MODEL_SCORES_CSV,
    random_state=RANDOM_STATE,
)
scores

## 7. Results/figures

In [ ]:
print(f"Saved model scores to {MODEL_SCORES_CSV}")
print(f"Saved confusion matrices to {FIGURES_DIR}")

## 8. Notes for report

- Report balanced accuracy and F1 alongside accuracy because class balance may vary across splits.
- Treat this as a classical ML baseline before CNN experiments.
- The split groups by `sample + area` to reduce repeated-area leakage.